
# Boston 311 Intake Assistant


## Step 1: Build the Request Type Catalog
1. Purpose: Build a clean list of request types (the "catalog") that the Intake Assistant will classify resident messages into.

2. Data : Used the 2024 311 data only(the catalog does not need 2023 data).

3. Team decisions reflected :
- In-scope department :  PWDx, BTDT, PARK, ISD.
- Keep the ORIGINAL request type names, so our data stays consistent with the Data & Forecast part.
- Exclude NO types (e.g., General Comments stays in). Many of these requests were routed to field queues, and the volume is small (~1,000).
- The catalog uses the top 40 types, which cover about 95% of requests.
- Messages outside the catalog are classified as "Other" (the LLM still
picks a department), and truly unclear messages as "Unknown".

4. Outputs(saved to Google Drive):
- type_list.csv: all request types with their main department/queue
- catalog_top40.csv : the top 40 types used by the Intake Assistant



In [ ]:
# 1. mount Google Drive and set file paths

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from typing import Literal
from pydantic import BaseModel
from google import genai
from google.colab import userdata
import re
import os, ast

BASE_DIR = "/content/drive/MyDrive/IE5374_311_project"
RAW_CSV = f"{BASE_DIR}/311_2024.csv"   # Update to the actual downloaded file name

# Only the columns needed for the catalog. Reading a subset of columns
# keeps memory usage low for this large file.
#   type        : request type (the label the Intake Assistant predicts)
#   department  : department that handled the request
#   queue       : team queue within the department
#   case_title  : free-text title (used later to write type descriptions)
COLUMNS = ["type", "department", "queue", "case_title"]

# The four departments in scope for this project
DEPARTMENTS = ["PWDx", "BTDT", "PARK", "ISD"]

# Number of request types to keep in the catalog
TOP_N = 40

# Reuse saved results (evaluation, synthetic messages, classification) from
# Google Drive instead of calling the API again. Re-running these steps takes
# about 30 minutes and creates new synthetic messages, which would change the
# numbers shared with the pipeline. Set to False to run everything again.
REUSE_SAVED = True


# 2. Load the data and keep only the four in-scope departments
df = pd.read_csv(RAW_CSV, usecols=COLUMNS)
df = df[df["department"].isin(DEPARTMENTS)]

print(f"Requests in the four departments: {len(df):,}")
print(f"Unique request types: {df['type'].nunique()}")


# 3. Assign one main department to each request type
# The same type can appear under several departments (e.g., Parking
# Enforcement appears in all four), most likely because some requests were
# reassigned or misrouted. We treat the department that handled the most
# requests of that type as the "correct" department.

# Count requests for every (type, department) pair
td = df.groupby(["type", "department"]).size().reset_index(name="count")

# Total requests per type, across all departments
total = td.groupby("type")["count"].sum()

# For each type, the row with the highest count = its main department
main = td.loc[td.groupby("type")["count"].idxmax()].set_index("type")

type_list = pd.DataFrame({
    "department": main["department"],          # main department
    "count": total,                            # total requests of this type
    "dept_share": main["count"] / total,       # share handled by the main dept
}).reset_index().sort_values("count", ascending=False).reset_index(drop=True)

# Note: only 13 of 158 types have dept_share below 0.8, and most of them
# have very few requests, so this rule works well for nearly all types.


# 4. Assign one main queue to each request type
# The queue is chosen ONLY among requests handled by the type's main
# department. Choosing across all departments caused mismatches
# (e.g., department = PWDx but queue = ISD_Health).
#
# Note: some types are split across district queues
# (e.g., "Requests for Street Cleaning" -> District 1C, District 03, ...).
# The queue depends on location, not on the message content, so we use the
# most common queue as the default.
dq = df.merge(type_list[["type", "department"]], on=["type", "department"])
main_queue = dq.groupby("type")["queue"].agg(lambda x: x.value_counts().index[0])
type_list["main_queue"] = type_list["type"].map(main_queue)


# 5. Select the top types for the catalog
# Cumulative share of requests, from the most common type downward.
# Result on 2024 data: top 27 types cover 90%, top 40 cover 95%.
type_list["cum_share"] = type_list["count"].cumsum() / type_list["count"].sum()

catalog = type_list.head(TOP_N).copy()

print(f"Catalog size: {len(catalog)} types")
print(f"Share of requests covered: {catalog['cum_share'].iloc[-1]:.1%}")


# 6. Save the outputs to Google Drive
type_list.to_csv(f"{BASE_DIR}/type_list.csv", index=False)
catalog.to_csv(f"{BASE_DIR}/catalog_top40.csv", index=False)

catalog[["type", "department", "main_queue", "count", "dept_share"]]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Requests in the four departments: 276,417
Unique request types: 158
Catalog size: 40 types
Share of requests covered: 95.0%


,type,department,main_queue,count,dept_share
0,Parking Enforcement,BTDT,BTDT_Parking Enforcement,63064,0.998557
1,Requests for Street Cleaning,PWDx,PWDx_District 1C: Downtown,20564,0.978798
2,Improper Storage of Trash (Barrels),PWDx,PWDx_Code Enforcement,20340,0.996706
3,CE Collection,PWDx,PWDx_District 1C: Downtown,18011,0.998112
4,Schedule a Bulk Item Pickup,PWDx,PWDx_Schedule a Bulk Item Pickup,11828,1.000000
5,Missed Trash/Recycling/Yard Waste/Bulk Item,PWDx,PWDx_Missed Trash\Recycling\Yard Waste\Bulk Item,10588,0.999622
6,Poor Conditions of Property,PWDx,PWDx_Code Enforcement,8706,0.992764
7,Mattress_Pickup,PWDx,PWDx_WM_Mattress_Pickup,8533,1.000000
8,Tree Maintenance Requests,PARK,PARK_Tree Maintenance Request,8355,0.965051
9,Request for Pothole Repair,PWDx,PWDx_Roadway Repair_ARP_Resurfacing,8029,0.993150


In [ ]:
# 7. Collect example case titles for each catalog type
# The 3 most common case_title values per type help us understand what each
# type means, especially for unclear names like "CE Collection".
top_types = catalog["type"].tolist()
examples = (df[df["type"].isin(top_types)]
              .groupby("type")["case_title"]
              .agg(lambda x: x.value_counts().head(3).index.tolist()))
catalog["examples"] = catalog["type"].map(examples)

pd.set_option("display.max_colwidth", None)
catalog[["type", "department", "examples"]]

,type,department,examples
0,Parking Enforcement,BTDT,[Parking Enforcement]
1,Requests for Street Cleaning,PWDx,"[Requests for Street Cleaning, Street Cleaning, Pick up Dead Animal]"
2,Improper Storage of Trash (Barrels),PWDx,"[Improper Storage of Trash (Barrels), Ground Maintenance: Polcari Park - BPRD, Ground Maintenance: Thetford Street Playground - BPRD]"
3,CE Collection,PWDx,"[CE Collection, Ground Maintenance: Chester Park - BPRD, Ground Maintenance: Charter St. Park - BPRD]"
4,Schedule a Bulk Item Pickup,PWDx,[Schedule a Bulk Item Pickup]
5,Missed Trash/Recycling/Yard Waste/Bulk Item,PWDx,"[Missed Trash: District 07, Missed Trash: District 02, Missed Trash: District 06]"
6,Poor Conditions of Property,PWDx,"[Poor Conditions of Property, Student Move-in CE Violation, WEEDS]"
7,Mattress_Pickup,PWDx,[Mattress Pickup]
8,Tree Maintenance Requests,PARK,"[Tree Maintenance Requests, Tree Maintenance, WEEDS]"
9,Request for Pothole Repair,PWDx,"[Request for Pothole Repair, Pothole, SINKHOLE]"


In [ ]:
# 8. Add plain-language descriptions to the catalog

# Why: the RAG step finds candidate types by comparing the resident's message
# with each type's text. Short internal names (e.g., "CE Collection") match
# poorly, so each type gets a 1-2 sentence description written the way a
# resident would describe the problem.
#
# Writing rules used:
#   - Use everyday words a resident would use, not city jargon.
#   - Make easily confused types clearly different
#     (e.g., Pest Infestation = inside the home, Rodent Activity = outdoors).
#
# Descriptions checked against the data:
#   - "CE Collection" and "Sticker Request" were confirmed using the
#     reason and closure_reason columns.
#   - Bulk Item Pickup and CE Collection were reworded after test v1
#     (scheduling a pickup from home vs. items already left on the street).
#
# Note: dictionary keys must match the type names in the data EXACTLY,
# including double spaces (e.g., "New Sign  Crosswalk or Pavement Marking").

DESCRIPTIONS = {
    # --- Transportation (BTDT) ---
    "Parking Enforcement":
        "A car is parked illegally, such as blocking a driveway, fire hydrant, "
        "crosswalk, or bike lane, parked in a no-parking or resident-only zone, "
        "or double-parked.",
    "Abandoned Vehicles":
        "A car, scooter, moped, or motorcycle has been left on the street for a "
        "long time and seems abandoned.",
    "Abandoned Bicycle":
        "A bicycle has been left locked or abandoned on a rack, pole, or "
        "sidewalk for a long time.",
    "Sign Repair":
        "An existing street or traffic sign is damaged, knocked down, faded, "
        "or missing.",
    "Traffic Signal Inspection":
        "A traffic light is not working, stuck, flashing, damaged, or the "
        "timing seems wrong.",
    "New Sign  Crosswalk or Pavement Marking":
        "I would like a new traffic sign, crosswalk, or road marking, or the "
        "painted lines on the road are faded.",
    "General Comments For a Program or Policy":
        "I want to give feedback, a complaint, or a compliment about a city "
        "program, policy, or employee.",

    # --- Public Works (PWDx): trash and recycling ---
    "Improper Storage of Trash (Barrels)":
        "Trash bags or barrels are left out on the sidewalk on the wrong day, "
        "without lids, overflowing, or not stored properly by a property.",
     "CE Collection":
        "Mattresses, furniture, or bags already left out or abandoned on the "
        "street or sidewalk, often during move-out season, that the street "
        "cleaning crew needs to pick up.",
     "Schedule a Bulk Item Pickup":
        "I want to schedule a pickup from my home for large household items "
        "like furniture or appliances that I am getting rid of.",
     "Schedule a Bulk Item Pickup SS":
        "I want to schedule a pickup from my home for large household items "
        "like furniture or appliances (self-service request).",
    "Missed Trash/Recycling/Yard Waste/Bulk Item":
        "My trash, recycling, yard waste, or bulk item was not picked up on the "
        "scheduled collection day.",
    "Mattress_Pickup":
        "I need to get rid of a mattress or box spring and want it picked up.",
    "Recycling Cart Return":
        "I want to return my city recycling cart.",
    "Request for Recycling Cart":
        "I need a new or replacement recycling cart.",
    "Sticker Request":
        "I need recycling stickers or a trash and recycling guide delivered "
        "to my home.",
    "Illegal Dumping":
        "Someone illegally dumped trash, furniture, tires, or construction "
        "debris on a street, lot, or public space.",
    "Empty Litter Basket":
        "A public trash can on the street is full or overflowing and needs to "
        "be emptied.",

    # --- Public Works (PWDx): streets, sidewalks, lights, snow ---
    "Requests for Street Cleaning":
        "The street or gutter is dirty and needs sweeping, for example litter, "
        "broken glass, leaves, or debris on the road.",
    "Request for Pothole Repair":
        "There is a pothole, hole, crack, or sinkhole in the road surface.",
    "Sidewalk Repair (Make Safe)":
        "The sidewalk is broken, cracked, uneven, or has loose bricks, creating "
        "a tripping hazard.",
    "Street Light Outages":
        "A street light is out, flickering, or not working.",
    "General Lighting Request":
        "A lighting problem other than a single outage, such as a damaged light "
        "pole, exposed wires, or a request for new or brighter lighting.",
    "Unshoveled Sidewalk":
        "The sidewalk in front of a property has not been cleared of snow or "
        "ice.",
    "Request for Snow Plowing":
        "The street has not been plowed and is still covered in snow.",
    "Pick up Dead Animal":
        "There is a dead animal on the street, sidewalk, or public area that "
        "needs to be removed.",
    "PWD Graffiti":
        "There is graffiti on public property, such as a wall, sign, or "
        "utility box, that needs to be removed.",
    "Poor Conditions of Property":
        "A private property or lot looks neglected, with overgrown weeds, "
        "trash, or debris in the yard.",
    "Contractor Complaints":
        "A contractor working on the street or sidewalk left a mess, damaged "
        "the road, or did poor work.",

    # --- Parks & Recreation (PARK) ---
    "Tree Maintenance Requests":
        "A street tree needs pruning or care, such as low branches blocking "
        "signs or lights, or a dead tree that should be removed (not an "
        "emergency).",
    "Tree Emergencies":
        "A tree or large branch has fallen or is about to fall, blocking a road "
        "or sidewalk, or hitting wires or property.",
    "New Tree Requests":
        "I would like the city to plant a new street tree.",
    "Ground Maintenance":
        "A park or playground needs upkeep, such as mowing grass, picking up "
        "litter, or cleaning the grounds.",
    "Equipment Repair":
        "Playground or park equipment, such as swings, benches, fences, or "
        "courts, is broken and needs repair.",

    # --- Inspectional Services (ISD) ---
    "Rodent Activity":
        "I see rats or mice, or signs of rodents like burrows or droppings, "
        "outdoors on the street, in a yard, or around a building.",
    "Pest Infestation - Residential":
        "There are cockroaches, bedbugs, mice, or other pests inside my home "
        "or apartment.",
    "Unsatisfactory Living Conditions":
        "My apartment or rental home has problems like leaks, mold, broken "
        "fixtures, or unsafe conditions that the landlord has not fixed.",
    "Heat - Excessive  Insufficient":
        "My apartment has no heat, not enough heat, or is too hot, and the "
        "landlord has not fixed it.",
    "Contractors Complaint":
        "A contractor working on a building is causing problems, such as "
        "unsafe work, noise, or work that breaks building rules.",
    "Work w/out Permit":
        "Someone is doing construction or renovation work on a building "
        "without a permit.",
}

# Attach descriptions to the catalog
catalog["description"] = catalog["type"].map(DESCRIPTIONS)

# Safety check: every catalog type must have a description.
# If anything prints here, the type name in the dictionary does not match the
# data exactly (check spaces and punctuation).
missing = catalog.loc[catalog["description"].isna(), "type"].tolist()
print("Types without a description:", missing)


# Save the catalog with descriptions (examples were only for writing these)
catalog = catalog.drop(columns=["examples"])
catalog.to_csv(f"{BASE_DIR}/catalog_top40.csv", index=False)

Types without a description: []


## Step 2: Embedding Retrieval(RAG)
1. Purpose: Given a resident's message, find the most similar request types in the
 catalog. These candidates are later passed to the LLM, which picks the final answer. Narrowing 40 types down to a few candidates makes the LLM more accurate and keeps the prompt short.
2. Libraries : We use sentence-transformers here because it runs locally in Colab,  free, no API key,and no rate limits.
Also 'all-MiniLM-L6-v2' is small and fast, and works well for short English sentences like resident messages.
3. Input : catalog_top40.csv (from Step 1, with descriptions)

4. Output: retrieve_candidates(message, k) -> top-k candidate types

In [ ]:
!pip install -q sentence-transformers

In [ ]:
# 1. Setup

BASE_DIR = "/content/drive/MyDrive/IE5374_311_project"
catalog = pd.read_csv(f"{BASE_DIR}/catalog_top40.csv")


# 2. Embed the catalog
# Each type is embedded as "type name: description". The description carries
# most of the meaning, written in residents' everyday words; the type name
# adds extra keywords.

model = SentenceTransformer("all-MiniLM-L6-v2")

catalog["embed_text"] = catalog["type"] + ": " + catalog["description"]

# normalize_embeddings=True scales every vector to length 1, so a simple dot
# product equals cosine similarity (1 = same meaning, 0 = unrelated).

catalog_vectors = model.encode(
    catalog["embed_text"].tolist(),
    normalize_embeddings=True,
)
print("Catalog embeddings:", catalog_vectors.shape)   # (40, 384)


# 3. Retrieval function

def retrieve_candidates(message, k=5):
    """
    Return the k catalog types most similar to a resident's message.

    Args:
        message (str): the resident's plain-language message
        k (int): number of candidates to return

    Returns:
        DataFrame with type, department, main_queue, description, and
        similarity score, sorted from most to least similar.
    """
    message_vector = model.encode([message], normalize_embeddings=True)[0]
    scores = catalog_vectors @ message_vector          # cosine similarity
    top_idx = np.argsort(scores)[::-1][:k]             # highest scores first

    result = catalog.iloc[top_idx][["type", "department", "main_queue", "description"]].copy()
    result["score"] = scores[top_idx].round(3)
    return result.reset_index(drop=True)


# 4. Quick check with a few sample messages
# The expected answer should appear among the top candidates. If it does not,
# the description for that type probably needs better wording.

samples = [
    "There's a big hole on Centre St near the T stop",
    "Someone parked in front of the fire hydrant on my street",
    "There are mice in my kitchen and the landlord won't do anything",
    "A tree branch fell and is blocking the sidewalk",
    "My trash wasn't picked up this morning",
]

for msg in samples:
    print(f"\nMessage: {msg}")
    print(retrieve_candidates(msg, k=5)[["type", "department", "score"]].to_string(index=False))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Catalog embeddings: (40, 384)

Message: There's a big hole on Centre St near the T stop
                      type department  score
Request for Pothole Repair       PWDx  0.388
              PWD Graffiti       PWDx  0.215
               Sign Repair       BTDT  0.181
  Request for Snow Plowing       PWDx  0.178
             CE Collection       PWDx  0.174

Message: Someone parked in front of the fire hydrant on my street
               type department  score
Parking Enforcement       BTDT  0.496
    Illegal Dumping       PWDx  0.311
       PWD Graffiti       PWDx  0.291
 Abandoned Vehicles       BTDT  0.288
Unshoveled Sidewalk       PWDx  0.274

Message: There are mice in my kitchen and the landlord won't do anything
                            type department  score
  Pest Infestation - Residential        ISD  0.634
                 Rodent Activity        ISD  0.544
Unsatisfactory Living Conditions        ISD  0.431
  Heat - Excessive  Insufficient        ISD  0.406
             Pick 

## Step 3: LLM Classification (Gemini)

1. Purpose: Take a resident's message, retrieve candidate types (Step 2), and let the LLM pick the final request type and department as structured JSON. If the message is unclear, the LLM also writes a follow-up question.

2. Design decisions: The LLM chooses ONLY from the retrieved candidates, plus:
- "Other"   : the request is not in the catalog (the LLM still picks a department, so no message is lost from the dept share)
- "Unknown" : the message is too unclear to choose a department
- The queue is NOT chosen by the LLM. It is looked up from the catalog (main_queue), because queues depend on location, not message content.
- If the chosen type is in the catalog, its department is taken from the catalog, so the type and department are always consistent.

3. Requirements:
- Run Step 2 first in this notebook, so that `catalog` and `retrieve_candidates()` exist.
- Store the API key in Colab Secrets as GOOGLE_API_KEY(never write the key in the code, since the code goes to GitHub).

In [ ]:
# 1. Setup: Gemini client
# !pip install -q google-genai   # uncomment if not installed


client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))

# Model names change over time, so we pin a specific Flash-Lite model
#(free tier) to keep the evaluation results reproducible.

MODEL_NAME = "gemini-3.1-flash-lite"


In [ ]:
# 2. Output schema (structured output)

# Gemini is forced to return JSON matching this schema, so the output can be
# used directly in code without parsing free text.

class IntakeResult(BaseModel):
    request_type: str                                   # a candidate type, "Other", or "Unknown"
    department: Literal["PWDx", "BTDT", "PARK", "ISD", "Unknown"]
    confidence: Literal["high", "medium", "low"]
    needs_follow_up: bool                               # True if the message is unclear
    follow_up_question: str                             # empty string if not needed
    reason: str                                         # one short sentence explaining the choice


In [ ]:
# 3. Prompt

# The system instruction gives the LLM its role, short department
# descriptions (to choose a department for "Other" messages), and clear rules.

SYSTEM_INSTRUCTION = """
You are the intake assistant for Boston 311, the city's non-emergency
service request system. Read a resident's message and classify it.

Departments:
- PWDx (Public Works): trash and recycling collection, bulk item pickup,
  street cleaning, potholes, sidewalks, street lights, snow plowing,
  graffiti on public property, dead animals on the street, and code
  enforcement for private property (improper trash storage, neglected
  yards, parking on lawns or yards).
- BTDT (Transportation): parking enforcement, abandoned vehicles or bikes,
  traffic signs, traffic signals, crosswalks and road markings.
- PARK (Parks & Recreation): street trees, parks, playgrounds, park equipment.
- ISD (Inspectional Services): housing conditions, heat, pests or rodents,
  building safety, construction work without a permit.

Rules:
1. Choose request_type from the candidate list ONLY, using the exact name.
2. If none of the candidates fit, use "Other" and still choose the most
   likely department.
3. If the message is too unclear to choose a department, use "Unknown"
   for both request_type and department.
4. If the message is unclear or missing key details (for example, what the
   problem is), set needs_follow_up to true and write one short, friendly
   follow-up question. Otherwise, use an empty string.
5. Keep the reason to one short sentence.
6. If the message is feedback, a complaint, or a compliment about a city
   policy, program, or employee (not a problem to fix), choose a
   "General Comments" type if one is among the candidates; otherwise use
   "Other" and choose the department the feedback is about.
"""


def build_prompt(message, candidates):
    """Combine the resident's message with the retrieved candidate types."""
    candidate_lines = "\n".join(
        f"- {row.type} ({row.department}): {row.description}"
        for row in candidates.itertuples()
    )
    return f"Resident message:\n{message}\n\nCandidate request types:\n{candidate_lines}"

In [ ]:
# 4. Classification function
# Lookups from the catalog, used to keep type, department, and queue consistent

TYPE_TO_DEPT = dict(zip(catalog["type"], catalog["department"]))
TYPE_TO_QUEUE = dict(zip(catalog["type"], catalog["main_queue"]))


def classify_message(message, k=5):
    """
    Classify one resident message.

    Steps:
        1. Retrieve the top-k candidate types (embedding search).
        2. Ask Gemini to choose one, returning JSON (IntakeResult).
        3. Clean up the result:
           - an invalid type name becomes "Other"
           - a catalog type gets its department and queue from the catalog

    Returns:
        dict with message, request_type, department, queue, confidence,
        needs_follow_up, follow_up_question, reason, and candidates.
    """
    candidates = retrieve_candidates(message, k=k)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=build_prompt(message, candidates),
        config={
            "system_instruction": SYSTEM_INSTRUCTION,
            "response_mime_type": "application/json",
            "response_schema": IntakeResult,
            "temperature": 0,          # same input -> same answer (needed for evaluation)
            "thinking_config": {"thinking_level": "low"},   # less internal reasoning = faster
        },
    )
    result = response.parsed.model_dump()

    # Keep the LLM's original answer for debugging and error analysis
    result["raw_type"] = result["request_type"]

    # Safety check: match the LLM's answer to an allowed name, ignoring
    #   - differences in spaces and capitalization
    #   - a department code in parentheses copied from the prompt
    # The exact catalog name is restored so the type matches the data.

    def normalize(name):
        name = re.sub(r"\s*\((PWDx|BTDT|PARK|ISD)\)\s*$", "", str(name))
        return " ".join(name.split()).lower()

    allowed = list(candidates["type"]) + ["Other", "Unknown"]
    allowed_lookup = {normalize(n): n for n in allowed}
    result["request_type"] = allowed_lookup.get(normalize(result["request_type"]), "Other")

    # Unknown department means the whole message is unclassifiable
    if result["department"] == "Unknown":
        result["request_type"] = "Unknown"

    # Catalog types: department and queue come from the catalog
    if result["request_type"] in TYPE_TO_DEPT:
        result["department"] = TYPE_TO_DEPT[result["request_type"]]
        result["queue"] = TYPE_TO_QUEUE[result["request_type"]]
    else:
        result["queue"] = None     # "Other" / "Unknown" have no catalog queue

    result["message"] = message
    result["candidates"] = candidates["type"].tolist()   # kept for error analysis
    return result

In [ ]:
# 5. Quick check
# Includes one clear message, one outside the catalog, one unclear message,
# and one general feedback message.

samples = [
    "There's a big hole on Cent St near the T stop",
    "I found a used needle on the playground near my house",   # likely "Other"
    "It's really bad on my street, please do something",       # unclear
    "The driver who collected our trash was so kind, thank you!",
]

for msg in samples:
    r = classify_message(msg)
    print(f"\nMessage   : {r['message']}")
    print(f"Type      : {r['request_type']}  |  Dept: {r['department']}  |  Queue: {r['queue']}")
    print(f"Confidence: {r['confidence']}  |  Reason: {r['reason']}")
    if r["needs_follow_up"]:
        print(f"Follow-up : {r['follow_up_question']}")


Message   : There's a big hole on Cent St near the T stop
Type      : Request for Pothole Repair  |  Dept: PWDx  |  Queue: PWDx_Roadway Repair_ARP_Resurfacing
Confidence: high  |  Reason: The resident is reporting a hole in the road surface, which falls under pothole repair.

Message   : I found a used needle on the playground near my house
Type      : Other  |  Dept: PARK  |  Queue: None
Confidence: high  |  Reason: The report concerns a safety hazard located within a park or playground area.

Message   : It's really bad on my street, please do something
Type      : Unknown  |  Dept: Unknown  |  Queue: None
Confidence: low  |  Reason: The resident's message is too vague to determine the nature of the problem or the appropriate department.
Follow-up : Could you please clarify what specific issue you are experiencing on your street so we can direct this to the right department?

Message   : The driver who collected our trash was so kind, thank you!
Type      : Other  |  Dept: PWDx  |  

## Step 4: Evaluation on the Test Set
1. Purpose: Run every test message through the Intake Assistant and measure:
- Type accuracy       : predicted type is one of the correct types
- Department accuracy : predicted department matches the correct one
(this is what the pipeline's dept share uses)
- Retrieval hit rate  : a correct type appears in the top-k candidates
(if not, the LLM could never pick it)
- Results are also broken down by category (clear, same-dept group,
cross-dept pair, other, feedback, unknown) for error analysis.

2. Requirements:
- Run Steps 2 and 3 first in the same notebook, so that `retrieve_candidates()` and `classify_message()` exist.
- test_set.csv is in the project folder on Google Drive. correct_types may contain several accepted answers separated by ";".


In [ ]:
# 1. Load the test set
import time
import pandas as pd
from google.genai import errors

TEST_NAME = "test_set_v2"     # "test_set" (v1) or "test_set_v2"
test_set = pd.read_csv(f"{BASE_DIR}/{TEST_NAME}.csv")

print(f"Test messages: {len(test_set)}")
print(test_set["category"].value_counts())


EVAL_PATH = f"{BASE_DIR}/evaluation_{TEST_NAME}.csv"

if REUSE_SAVED and os.path.exists(EVAL_PATH):
    # Load the saved results instead of calling the API again
    results = pd.read_csv(EVAL_PATH)
    results["candidates"] = results["candidates"].apply(ast.literal_eval)
    print(f"Loaded saved results: {EVAL_PATH}")
else:
    # 2. Run the Intake Assistant on every test message
    # The free tier limits requests per minute, so we pause between calls.
    # If a call fails (busy server or rate limit), we wait and retry a few times.
    # The model name is recorded so the report states exactly what was evaluated.

    PAUSE_SECONDS = 6      # pause between messages (free-tier rate limit)
    MAX_RETRIES = 3        # retries per message on a temporary error

    def classify_with_retry(message):
        """Call classify_message(); on a temporary API error, wait and retry."""
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                return classify_message(message)
            except errors.APIError as e:
                wait = 20 * attempt
                print(f"  API error {e.code} (attempt {attempt}), waiting {wait}s...")
                time.sleep(wait)
        return None        # still failing after all retries

    predictions = []
    for row in test_set.itertuples():
        print(f"[{row.id}/{len(test_set)}] {row.message[:60]}")
        result = classify_with_retry(row.message)
        if result is None:
            result = {"request_type": "ERROR", "department": "ERROR", "queue": None,
                      "confidence": None, "needs_follow_up": None,
                      "follow_up_question": None, "reason": None, "candidates": []}
        result["id"] = row.id
        predictions.append(result)
        time.sleep(PAUSE_SECONDS)

    pred_df = pd.DataFrame(predictions)
    results = test_set.merge(
        pred_df[["id", "request_type", "department", "queue", "confidence",
                 "needs_follow_up", "follow_up_question", "reason", "candidates"]],
        on="id",
    ).rename(columns={"request_type": "pred_type", "department": "pred_department"})

    results["model"] = MODEL_NAME
    print("\nErrors (calls that failed):", (results["pred_type"] == "ERROR").sum())


Test messages: 38
category
clear              21
same-dept group     6
cross-dept pair     4
other               3
feedback            2
unknown             2
Name: count, dtype: int64
[1/38] A car has been double parked on Tremont St blocking traffic 
[2/38] Someone is parked in the handicap spot without a placard aga
[3/38] The one-way sign at the end of my street is missing
[4/38] The walk signal at the intersection by the library never com
[5/38] The lane lines on Columbia Rd are completely faded
[6/38] Trash day was yesterday and our barrels are still full
[7/38] There's a sinkhole forming in the road on Hyde Park Ave
[8/38] The bricks on the sidewalk outside the library are loose and
[9/38] The light pole on our corner is leaning and wires are hangin
[10/38] The sidewalk on Centre St is a sheet of ice, nobody shoveled
[11/38] There's a dead cat on the sidewalk near the school
[12/38] Someone tagged the bus shelter with spray paint
[13/38] Our blue recycling bin cracked, we need a

In [ ]:
# 3. Score each message

# correct_types is a ";"-separated list of accepted answers.
results["correct_type_list"] = results["correct_types"].str.split(";")

# Type is correct if the prediction is any accepted answer
results["type_correct"] = results.apply(
    lambda r: r["pred_type"] in r["correct_type_list"], axis=1)

# Department is correct if it matches exactly (includes "Unknown")
results["dept_correct"] = results["pred_department"] == results["correct_department"]

# Retrieval hit: only meaningful when the answer is a catalog type
# (not for "Other" or "Unknown" answers, which are never retrieved).
def retrieval_hit(r):
    catalog_answers = [t for t in r["correct_type_list"] if t not in ("Other", "Unknown")]
    if not catalog_answers:
        return None
    return any(t in r["candidates"] for t in catalog_answers)

results["retrieval_hit"] = results.apply(retrieval_hit, axis=1)


# 4. Summary: overall and by category
print(f"Model: {MODEL_NAME}")
print(f"Type accuracy       : {results['type_correct'].mean():.1%}")
print(f"Department accuracy : {results['dept_correct'].mean():.1%}")
hits = results["retrieval_hit"].dropna()
print(f"Retrieval hit rate  : {hits.mean():.1%}  (top-5, catalog answers only, n={len(hits)})")

by_category = (results.groupby("category")
                 .agg(n=("id", "size"),
                      type_acc=("type_correct", "mean"),
                      dept_acc=("dept_correct", "mean"))
                 .round(2))
by_category


# 5. Error analysis: every message the assistant got wrong
# These cases go into the report: what was confused, and why.

errors_df = results[~results["type_correct"] | ~results["dept_correct"]]
pd.set_option("display.max_colwidth", None)
errors_df[["id", "category", "message", "correct_types", "pred_type",
           "correct_department", "pred_department", "reason"]]


# 6. Save the results to Google Drive
# Saved so the evaluation does not need to be re-run (it takes several
# minutes on the free tier) and can be used in the report and demo.
results.drop(columns=["correct_type_list"]).to_csv(
    f"{BASE_DIR}/evaluation_{TEST_NAME}.csv", index=False)

Model: gemini-3.1-flash-lite
Type accuracy       : 94.7%
Department accuracy : 100.0%
Retrieval hit rate  : 97.0%  (top-5, catalog answers only, n=33)


In [ ]:
display(by_category)

display(errors_df[["id", "category", "message", "correct_types", "pred_type",
                   "correct_department", "pred_department", "reason"]])

,n,type_acc,dept_acc
category,,,
clear,21,1.00,1.0
cross-dept pair,4,1.00,1.0
feedback,2,1.00,1.0
other,3,0.67,1.0
same-dept group,6,0.83,1.0
unknown,2,1.00,1.0


,id,category,message,correct_types,pred_type,correct_department,pred_department,reason
15,16,same-dept group,Can someone come take away our old desk and a bookshelf?,Schedule a Bulk Item Pickup;Schedule a Bulk Item Pickup SS,CE Collection,PWDx,PWDx,"The resident is requesting the removal of furniture, which falls under bulk item collection."
31,32,other,My neighbor parks his truck on the grass in his backyard every night,Other,Poor Conditions of Property,PWDx,PWDx,Parking on lawns or yards on private property falls under code enforcement handled by Public Works.


## Step 5: Hand-off to the Pipeline (August)
1. Purpose: Provide what the Model & Pipeline part needs from Intake:  1.(1)classify_messages(messages)  -> classify a stream of messages
(2)compute_dept_share(results)  -> observed department share
(3)Synthetic August message streams: "normal" and "parking surge"

2. Team decision (pipeline scenario):
(1) The synthetic stream represents ALL complaints received in August.
(2) Intake hands off the extra requests per department (extra_requests). The pipeline adds them to the forecast:
- adjusted demand = August forecast + extra_requests
- and warns about crew shortages for the following month.
(3) Message counts are converted to real scale with a scale factor per department, calibrated on the normal stream:
- scale factor = August 2025 forecast (department) / normal message count (department)
- With only 80 messages, one overall scale factor made small departments swing a lot (ISD +22% from a single extra message).
- So the normal stream adds 0 by design, and in the surge only BTDT gets extra requests. Classification accuracy is checked separately with the test sets (Step 4).

3. Why August 2024 (not the full year):
- The synthetic stream should match the same month as the forecast it is compared with. Demand is seasonal (e.g., Parks grows strongly in summer), so a full-year mix would not look like a normal August.

4. Other design decisions:
- Share denominator: the four departments only; "Unknown" is excluded.
- Parking surge = normal August stream + extra Parking Enforcement messages (doubled). The normal classification is reused.
- Each synthetic message keeps the type it was written for, so we can also check accuracy on the stream.

5. Requirements:
- Run Steps 1-3 first (df, type_list, catalog, DEPARTMENTS, RAW_CSV,
client, MODEL_NAME, classify_message must exist).
- Run time on the free tier: about 15 minutes in total.


In [ ]:
# 1. Batch classification: classify_messages()
import time
import pandas as pd
from pydantic import BaseModel
from google.genai import errors


def call_with_retry(fn, *args, max_retries=3, **kwargs):
    """Run an API call; on a temporary error (busy server, rate limit), wait and retry."""
    for attempt in range(1, max_retries + 1):
        try:
            return fn(*args, **kwargs)
        except errors.APIError as e:
            wait = 20 * attempt
            print(f"  API error {e.code} (attempt {attempt}), waiting {wait}s...")
            time.sleep(wait)
    return None


def classify_messages(messages, pause_seconds=6):
    """
    Classify a list of resident messages with the Intake Assistant.

    Returns:
        DataFrame with one row per message:
        message, request_type, department, queue, confidence, needs_follow_up
    """
    columns = ["message", "request_type", "department", "queue",
               "confidence", "needs_follow_up"]
    rows = []
    for i, msg in enumerate(messages, 1):
        result = call_with_retry(classify_message, msg)
        if result is None:                       # still failing after retries
            result = {"message": msg, "request_type": "ERROR", "department": "ERROR"}
        rows.append({c: result.get(c) for c in columns})

        if i % 10 == 0 or i == len(messages):
            print(f"  classified {i}/{len(messages)}")
        if i < len(messages):
            time.sleep(pause_seconds)
    return pd.DataFrame(rows, columns=columns)




In [ ]:
# 2. Observed department share: compute_dept_share()

def compute_dept_share(classified):
    """
    Share of classified messages per department (four departments only).
    "Unknown" and "ERROR" messages are excluded from the denominator.
    All four departments are always listed, even with 0 messages.

    Returns:
        DataFrame: department, observed_count, observed_share
    """
    valid = classified[classified["department"].isin(DEPARTMENTS)]
    counts = valid["department"].value_counts().reindex(DEPARTMENTS, fill_value=0)

    excluded = len(classified) - len(valid)
    if excluded:
        print(f"  excluded from share (Unknown/ERROR): {excluded}")

    return pd.DataFrame({
        "department": DEPARTMENTS,
        "observed_count": counts.values,
        "observed_share": (counts / counts.sum()).round(4).values,
    })

In [ ]:
# 3. August 2024 request mix

# Step 1 did not load dates, so we read only the columns needed here.
MONTH = 8                                   # August

dates = pd.read_csv(RAW_CSV, usecols=["type", "department", "open_dt"])
dates = dates[dates["department"].isin(DEPARTMENTS)]
# The raw file mixes date formats (with and without fractional seconds),
# so each row is parsed on its own format. Without format="mixed", about 40%
# of rows become NaT and the August mix is distorted.
dates["open_dt"] = pd.to_datetime(dates["open_dt"], format="mixed", errors="coerce")
print("Rows with unreadable dates:", dates["open_dt"].isna().sum())
month_df = dates[dates["open_dt"].dt.month == MONTH]

# Requests per type in August
month_counts = month_df["type"].value_counts()
type_share = month_counts / month_counts.sum()

print(f"August 2024 requests (four departments): {len(month_df):,}")


Rows with unreadable dates: 0
August 2024 requests (four departments): 27,337


In [ ]:
# 4. Synthetic message generator

class SyntheticBatch(BaseModel):
    messages: list[str]

GEN_INSTRUCTION = """
You write realistic messages that Boston residents send to the city's 311
non-emergency service in late summer (August). Vary the tone, length, and
details (street names, landmarks, how long the problem has lasted). Write
the way real residents write: sometimes short, sometimes frustrated or
polite. Never use the official category name in the message.
"""

def generate_messages(problem_description, n):
    """Ask the LLM for n different resident messages about one problem."""
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=f"Write {n} different resident messages about this problem:\n{problem_description}",
        config={
            "system_instruction": GEN_INSTRUCTION,
            "response_mime_type": "application/json",
            "response_schema": SyntheticBatch,
            "temperature": 1.0,             # varied wording (the classifier uses 0)
        },
    )
    return response.parsed.messages[:n]

In [ ]:
# 5. Plan the normal August stream

# Message counts per type follow the August 2024 volumes. Small types round
# to 0; the remaining slots go to types outside the catalog, which the
# assistant should classify as "Other".
N_TOTAL = 80                    # size of the stream (a sample of the month)
RANDOM_SEED = 42                # fixed seed so the plan can be re-created

TYPE_TO_DEPT_ALL = dict(zip(type_list["type"], type_list["department"]))
DESC = dict(zip(catalog["type"], catalog["description"]))

# Catalog types
catalog_plan = (type_share[type_share.index.isin(catalog["type"])] * N_TOTAL).round().astype(int)
catalog_plan = catalog_plan[catalog_plan > 0]

# Non-catalog types fill the remaining slots, sampled by August volume
n_other = max(N_TOTAL - catalog_plan.sum(), 0)
non_catalog = month_counts[~month_counts.index.isin(catalog["type"])]
other_plan = (non_catalog.index.to_series()
              .sample(n_other, weights=non_catalog.values, replace=True,
                      random_state=RANDOM_SEED)
              .value_counts())

print(f"Catalog types in stream: {len(catalog_plan)} ({catalog_plan.sum()} messages)")
print(f"Non-catalog messages   : {other_plan.sum()}")


Catalog types in stream: 27 (76 messages)
Non-catalog messages   : 4


In [ ]:
# 6. Generate the normal August stream
# Each row keeps the type it was written for:
#   source_type         : the real 311 type the message describes
#   expected_type       : the catalog type, or "Other" for non-catalog types
#   expected_department : the main department of the source type
def build_stream(plan, in_catalog, pause_seconds=6):
    records = []
    for source_type, n in plan.items():
        description = DESC[source_type] if in_catalog else source_type
        msgs = call_with_retry(generate_messages, description, int(n)) or []
        for m in msgs:
            records.append({
                "source_type": source_type,
                "expected_type": source_type if in_catalog else "Other",
                "expected_department": TYPE_TO_DEPT_ALL[source_type],
                "message": m,
            })
        time.sleep(pause_seconds)
    return records

NORMAL_PATH = f"{BASE_DIR}/synthetic_august.csv"

if REUSE_SAVED and os.path.exists(NORMAL_PATH):
    normal_stream = pd.read_csv(NORMAL_PATH)          # saved stream
else:
    normal_stream = pd.DataFrame(
        build_stream(catalog_plan, in_catalog=True) +
        build_stream(other_plan, in_catalog=False)
    ).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)   # shuffle
    normal_stream.to_csv(NORMAL_PATH, index=False)

print(f"Normal August stream: {len(normal_stream)} messages")


Normal August stream: 80 messages


In [ ]:
# 7. Generate the parking surge extra messages
# Scenario: Parking Enforcement requests double compared with a normal August.
SURGE_TYPE = "Parking Enforcement"
SURGE_MULTIPLIER = 2.0

n_extra = int(round(catalog_plan[SURGE_TYPE] * (SURGE_MULTIPLIER - 1)))
SURGE_PATH = f"{BASE_DIR}/synthetic_august_parking_surge_extra.csv"

if REUSE_SAVED and os.path.exists(SURGE_PATH):
    surge_extra = pd.read_csv(SURGE_PATH)             # saved messages
else:
    surge_extra = pd.DataFrame(build_stream(pd.Series({SURGE_TYPE: n_extra}), in_catalog=True))
    surge_extra.to_csv(SURGE_PATH, index=False)

print(f"Extra {SURGE_TYPE} messages: {len(surge_extra)}")


Extra Parking Enforcement messages: 15


In [ ]:
# 8. Classify the streams

NORMAL_CLS_PATH = f"{BASE_DIR}/classified_august.csv"
SURGE_CLS_PATH = f"{BASE_DIR}/classified_august_parking_surge.csv"

if REUSE_SAVED and os.path.exists(NORMAL_CLS_PATH) and os.path.exists(SURGE_CLS_PATH):
    normal_classified = pd.read_csv(NORMAL_CLS_PATH)  # saved classification
    surge_classified = pd.read_csv(SURGE_CLS_PATH)
    print("Loaded saved classification results")
else:
    print("Classifying normal August stream...")
    normal_classified = classify_messages(normal_stream["message"].tolist())

    print("Classifying surge extra messages...")
    extra_classified = classify_messages(surge_extra["message"].tolist())

    surge_classified = pd.concat([normal_classified, extra_classified], ignore_index=True)

    normal_classified.to_csv(NORMAL_CLS_PATH, index=False)
    surge_classified.to_csv(SURGE_CLS_PATH, index=False)


Classifying normal August stream...
  classified 10/80
  classified 20/80
  classified 30/80
  classified 40/80
  classified 50/80
  classified 60/80
  classified 70/80
  classified 80/80
Classifying surge extra messages...
  API error 500 (attempt 1), waiting 20s...
  classified 10/15
  classified 15/15


In [ ]:
# 9. Observed department shares (hand-off files for the pipeline)

share_normal = compute_dept_share(normal_classified)
share_surge = compute_dept_share(surge_classified)

share_normal.to_csv(f"{BASE_DIR}/observed_share_august.csv", index=False)
share_surge.to_csv(f"{BASE_DIR}/observed_share_august_parking_surge.csv", index=False)

# Reference only: August 2024 department share in the real data.
# The official base share comes from the Data & Forecast part.
data_counts = month_df["department"].value_counts().reindex(DEPARTMENTS)
reference = pd.DataFrame({
    "department": DEPARTMENTS,
    "data_share_aug2024": (data_counts / data_counts.sum()).round(4).values,
})

comparison = (reference
              .merge(share_normal[["department", "observed_share"]], on="department")
              .rename(columns={"observed_share": "normal_share"})
              .merge(share_surge[["department", "observed_share"]], on="department")
              .rename(columns={"observed_share": "surge_share"}))

# Ratio preview (the normal stream should be close to 1)
comparison["normal_ratio"] = (comparison["normal_share"] / comparison["data_share_aug2024"]).round(2)
comparison["surge_ratio"] = (comparison["surge_share"] / comparison["data_share_aug2024"]).round(2)
display(comparison)

,department,data_share_aug2024,normal_share,surge_share,normal_ratio,surge_ratio
0,PWDx,0.5998,0.6125,0.5158,1.02,0.86
1,BTDT,0.2458,0.2250,0.3474,0.92,1.41
2,PARK,0.0784,0.0750,0.0632,0.96,0.81
3,ISD,0.0760,0.0875,0.0737,1.15,0.97


In [ ]:
# 9b. Extra requests to add to the forecast (hand-off for the pipeline)
# Per-department scale factors, calibrated on the NORMAL August stream:
#     scale[dept] = August 2025 forecast[dept] / normal message count[dept]
#
# Why per department: with only 80 messages, small departments swing a lot
# (e.g., ISD had 7 messages instead of about 6, which became +22% with a
# single overall scale factor). Calibrating on the normal stream removes this
# sampling noise, like zeroing a scale before weighing.
#
# Result for the pipeline: adjusted demand = forecast + extra_requests
#   - normal stream : extra_requests = 0 for every department (by design)
#   - parking surge : only BTDT gets extra requests
# Classification accuracy is evaluated separately on the hand-written test sets.

AUGUST_FORECAST = {"PWDx": 16474, "BTDT": 7090, "PARK": 2360, "ISD": 1997}   # Data & Forecast, Aug 2025

normal_counts = share_normal.set_index("department")["observed_count"]
DEPT_SCALE = {d: AUGUST_FORECAST[d] / normal_counts[d] for d in DEPARTMENTS}

def estimate_volume(share_df):
    """Convert sample counts to real-scale requests and the extra over the forecast."""
    out = share_df[["department", "observed_count"]].copy()
    out["scale"] = out["department"].map(DEPT_SCALE).round(1)
    out["forecast"] = out["department"].map(AUGUST_FORECAST)
    out["estimated_requests"] = (out["observed_count"] * out["department"].map(DEPT_SCALE)).round().astype(int)
    out["extra_requests"] = out["estimated_requests"] - out["forecast"]   # add this to the forecast
    return out

volume_normal = estimate_volume(share_normal)
volume_surge = estimate_volume(share_surge)
volume_normal.to_csv(f"{BASE_DIR}/extra_requests_august.csv", index=False)
volume_surge.to_csv(f"{BASE_DIR}/extra_requests_august_parking_surge.csv", index=False)
display(volume_normal)
display(volume_surge)


,department,observed_count,scale,forecast,estimated_requests,extra_requests
0,PWDx,49,336.2,16474,16474,0
1,BTDT,18,393.9,7090,7090,0
2,PARK,6,393.3,2360,2360,0
3,ISD,7,285.3,1997,1997,0


,department,observed_count,scale,forecast,estimated_requests,extra_requests
0,PWDx,49,336.2,16474,16474,0
1,BTDT,33,393.9,7090,12998,5908
2,PARK,6,393.3,2360,2360,0
3,ISD,7,285.3,1997,1997,0


In [ ]:
# 10. Sanity check: accuracy on the synthetic normal stream

# Optimistic, because the messages were written by the same model family.
# Same-department groups are accepted, as in the test sets.
EQUIV_GROUPS = [
    {"Schedule a Bulk Item Pickup", "Schedule a Bulk Item Pickup SS"},
    {"CE Collection", "Mattress_Pickup", "Illegal Dumping"},
    {"Requests for Street Cleaning", "CE Collection"},
    {"Abandoned Vehicles", "Abandoned Bicycle"},
    {"Street Light Outages", "General Lighting Request"},
    {"Rodent Activity", "Pest Infestation - Residential"},
    {"Heat - Excessive  Insufficient", "Unsatisfactory Living Conditions"},
    {"Tree Maintenance Requests", "Tree Emergencies"},
]

def type_matches(expected, predicted):
    """True if the types are the same or belong to the same group."""
    if expected == predicted:
        return True
    return any(expected in g and predicted in g for g in EQUIV_GROUPS)

check = normal_stream.join(normal_classified[["request_type", "department"]])
check["type_ok"] = check.apply(lambda r: type_matches(r["expected_type"], r["request_type"]), axis=1)

print(f"Type accuracy (strict)      : {(check['request_type'] == check['expected_type']).mean():.1%}")
print(f"Type accuracy (with groups) : {check['type_ok'].mean():.1%}")
print(f"Department accuracy         : {(check['department'] == check['expected_department']).mean():.1%}")

Type accuracy (strict)      : 81.2%
Type accuracy (with groups) : 91.2%
Department accuracy         : 100.0%


## Step 6: Other Considerations (Employee vs. Resident Requests)

1. Purpose: About 42% of 2024 requests were logged by city employees, not residents. The Intake Assistant only receives residents' messages, so we compare the two groups to see what part of the demand Intake can actually observe.

2. Method:
- Employee requests: source is "Employee Generated" or "City Worker App"
- Resident requests: all other sources (app, phone call, self service)
- Compare by department: share of employee requests, on-time rate, and the most common request types

3. Findings:
- In the four departments, about 53% of requests are employee-generated (Transportation 62%, Parks 60%, Public Works 50%, ISD 40%).
- Many employee requests are issues crews find in the field: 66% of Parking Enforcement and almost all CE Collection requests. Resident requests include more service requests, such as mattress and bulk item pickups.
- On-time rates are almost the same for both groups (Transportation is about 50% for both), so the low on-time rate is not explained by who logs the request.
- The Intake Assistant only sees resident messages, about half of the demand (38% for Transportation). Our synthetic stream used the full mix, so a real system would need to add employee requests separately.

In [42]:
# Other considerations: employee-generated vs. resident requests
src = pd.read_csv(RAW_CSV, usecols=["type", "department", "source", "on_time"])
src = src[src["department"].isin(DEPARTMENTS)]

EMPLOYEE_SOURCES = ["Employee Generated", "City Worker App"]
src["requester"] = np.where(src["source"].isin(EMPLOYEE_SOURCES), "Employee", "Resident")
print(src["source"].value_counts())          # check the source labels first

# 1. Share of employee requests by department
display(pd.crosstab(src["department"], src["requester"], normalize="index").round(3))

# 2. On-time rate by department and requester
src["is_on_time"] = src["on_time"].eq("ONTIME")
display(src.pivot_table(index="department", columns="requester",
                        values="is_on_time", aggfunc="mean").round(3))

# 3. Top types for each requester
for who in ["Employee", "Resident"]:
    print(f"\nTop types - {who}")
    print(src[src["requester"] == who]["type"].value_counts().head(5))

source
Employee Generated      116828
Citizens Connect App     68476
Constituent Call         49551
City Worker App          30337
Self Service             11225
Name: count, dtype: int64


requester,Employee,Resident
department,,
BTDT,0.616,0.384
ISD,0.403,0.597
PARK,0.603,0.397
PWDx,0.496,0.504


requester,Employee,Resident
department,,
BTDT,0.501,0.497
ISD,0.747,0.709
PARK,0.950,0.904
PWDx,0.896,0.898



Top types - Employee
type
Parking Enforcement                    41618
CE Collection                          18011
Improper Storage of Trash (Barrels)    11466
Requests for Street Cleaning           10564
Tree Maintenance Requests               5833
Name: count, dtype: int64

Top types - Resident
type
Parking Enforcement                    21446
Requests for Street Cleaning           10000
Improper Storage of Trash (Barrels)     8874
Mattress_Pickup                         8448
Schedule a Bulk Item Pickup             8347
Name: count, dtype: int64


## Step 7: Demo UI
1. Purpose: A simple web page (Gradio) where a resident types a problem in their own words and sees where the request is sent.

2. What it shows:
- The department (name and code), request type, team, and confidence
- A follow-up question when the message is unclear
- The top 5 candidates from retrieval (RAG), to show how the answer was chosen

3. Note: The UI calls classify_message() from Step 3, so Steps 1-3 must be run first.


In [37]:
!pip install -q gradio

In [41]:
# Web UI for the Intake Assistant (for the demo)
import html
import gradio as gr

# Department names and colors shown to residents (codes kept for staff)
DEPT_INFO = {
    "PWDx": ("Public Works", "#2563eb"),
    "BTDT": ("Transportation", "#d97706"),
    "PARK": ("Parks & Recreation", "#16a34a"),
    "ISD": ("Inspectional Services", "#9333ea"),
    "Unknown": ("Not sure yet", "#6b7280"),
}

def result_card(r):
    """Build the result card shown to the resident."""
    name, color = DEPT_INFO.get(r["department"], (r["department"], "#6b7280"))
    team = r["queue"].split("_", 1)[-1] if r["queue"] else "To be assigned"
    follow = ""
    if r["needs_follow_up"] and r["follow_up_question"]:
        follow = (f'<div style="margin-top:12px;padding:10px 12px;border-radius:8px;'
                  f'background:rgba(217,119,6,0.15)"><b>We need a bit more detail:</b> '
                  f'{html.escape(r["follow_up_question"])}</div>')
    return f"""
<div style="border:1px solid rgba(128,128,128,0.35);border-left:6px solid {color};
            border-radius:10px;padding:16px 18px;line-height:1.6">
  <div style="font-size:0.85em;opacity:0.7">Your request was sent to</div>
  <div style="font-size:1.4em;font-weight:700;margin-bottom:8px">
    {name} <span style="font-size:0.6em;opacity:0.6">({r['department']})</span></div>
  <div><b>Request type:</b> {html.escape(r['request_type'])}</div>
  <div><b>Team:</b> {html.escape(team)}</div>
  <div><b>Confidence:</b> {r['confidence']}</div>
  <div style="margin-top:8px;opacity:0.8"><i>{html.escape(r['reason'])}</i></div>
  {follow}
</div>"""

def intake_ui(message):
    """Classify one message; return the result card and the RAG candidates."""
    if not message.strip():
        return "", None
    r = classify_message(message)
    candidates = retrieve_candidates(message)[["type", "department", "score"]]   # what the LLM saw
    return result_card(r), candidates

with gr.Blocks(title="Boston 311 Intake Assistant") as demo:
    gr.Markdown("## Boston 311 Intake Assistant\n"
                "Describe the problem in your own words. We'll send it to the right team.")
    with gr.Row():
        with gr.Column(scale=1):
            msg = gr.Textbox(label="What's the problem?", lines=4,
                             placeholder="e.g., There's a big hole on Centre St near the T stop")
            btn = gr.Button("Submit request", variant="primary")
            gr.Examples(examples=[["Someone parked in front of the fire hydrant on my street"],
                                  ["There are mice in my kitchen and the landlord won't do anything"],
                                  ["It's really bad on my street, please do something"]],
                        inputs=msg)
        with gr.Column(scale=1):
            card = gr.HTML()
            with gr.Accordion("How it was classified: top 5 candidates from retrieval (RAG)", open=False):
                cands = gr.Dataframe()
    btn.click(intake_ui, inputs=msg, outputs=[card, cands])
    msg.submit(intake_ui, inputs=msg, outputs=[card, cands])

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e3afa747a524f84cfc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
